In [1]:
from pyspark.sql import SparkSession


In [2]:

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [3]:
spark

In [4]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [6]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)

In [8]:
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [9]:
from pyspark.sql.functions import min as _min, max as _max

# TWÓJ KOD
# df.groupBy("category").agg(...).orderBy("category").show()

category_stats = (
    df.groupBy("category")
      .agg(
          _round(_sum("amount"), 2).alias("suma_PLN"),
          _round(_min("amount"), 2).alias("min_PLN"),
          _round(_max("amount"), 2).alias("max_PLN")
      )
      .orderBy("category")
)

category_stats.show(truncate=False)

+-----------+----------+-------+-------+
|category   |suma_PLN  |min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|9.0    |9999.0 |
|książki    |851382.08 |5.0    |9107.25|
|odzież     |849877.55 |5.0    |9696.63|
|żywność    |789514.43 |5.0    |6916.92|
+-----------+----------+-------+-------+



In [10]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)

In [11]:
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [12]:
hourly.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- liczba_tx: long (nullable = false)
 |-- suma_PLN: double (nullable = true)



In [13]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [14]:
from pyspark.sql.functions import window, count, sum as _sum

(
df.groupBy(window("timestamp", "30 minutes"), "store")
  .agg(
      count("tx_id").alias("liczba_tx"),
      _sum("amount").alias("suma_PLN"),
  )
  .orderBy("window", "store") 
  .show(truncate=False)
)

+------------------------------------------+--------+---------+------------------+
|window                                    |store   |liczba_tx|suma_PLN          |
+------------------------------------------+--------+---------+------------------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22000000002 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.4199999999 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58000000003 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.5899999999 |
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85000000012|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41000000006|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.05999999994|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.1699999999 |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Gdańsk  |619      |253364.94999999992|
|{20

In [15]:
from pyspark.sql.functions import window, count, sum as _sum, desc

(
    df.filter(df.store == "Kraków")
      .groupBy(window("timestamp", "1 hour"))
      .agg(
          count("tx_id").alias("liczba_tx"),
          _sum("amount").alias("suma_PLN")
      )
      .orderBy(desc("suma_PLN"))
      .show(truncate=False)
)

+------------------------------------------+---------+------------------+
|window                                    |liczba_tx|suma_PLN          |
+------------------------------------------+---------+------------------+
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|1169     |483309.85999999975|
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|821      |341327.82999999996|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|532      |201259.25999999995|
+------------------------------------------+---------+------------------+



In [16]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [17]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: Okna się nakładaja, wiec jedna transakcja wpada do wielu okien. 

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [18]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: pierwsze grupuje tylko po sklepie, drugie po czasie i sklepie 

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2

In [19]:
from pyspark.sql.functions import window, avg

(
    df.filter(df.store == "Gdańsk")
      .groupBy(window("timestamp", "1 hour"))
      .agg(avg("amount").alias("srednia_kwota"))
      .orderBy("srednia_kwota")
      .show(1)
)

+--------------------+-----------------+
|              window|    srednia_kwota|
+--------------------+-----------------+
|{2026-04-12 08:00...|395.0118407310706|
+--------------------+-----------------+
only showing top 1 row



In [20]:
from pyspark.sql.functions import count, col

(
    df.filter(
        (col("timestamp") >= "2026-04-12 09:00:00") &
        (col("timestamp") <  "2026-04-12 09:30:00")
    )
    .groupBy("category")
    .agg(count("tx_id").alias("liczba_tx"))
    .show()
)

+-----------+---------+
|   category|liczba_tx|
+-----------+---------+
|    książki|      622|
|     odzież|      605|
|elektronika|      611|
|    żywność|      567|
+-----------+---------+



In [21]:
from pyspark.sql.functions import window, count, desc

(
    df.groupBy(window("timestamp", "15 minutes"))
      .agg(count("tx_id").alias("liczba_tx"))
      .orderBy(desc("liczba_tx"))
      .show(1)
)

+--------------------+---------+
|              window|liczba_tx|
+--------------------+---------+
|{2026-04-12 09:15...|     1234|
+--------------------+---------+
only showing top 1 row



In [22]:
spark.stop()